# 12 — TF-IDF + LinearSVC (calibrado)

Pipeline com **RandomizedSearchCV** sobre o pool train+val (90%), seguido pelos 6 regimes (binario/multiclasse x fixed_split/cv_5fold/test_set). O test set (10%) so e tocado no regime `test_set`.

## 0. Bootstrap (Colab + local)

In [ ]:
import subprocess
import sys
import zipfile
from pathlib import Path


def _run(cmd: list[str], description: str) -> None:
    """Run a subprocess command, print stdout/stderr, raise on non-zero exit."""
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr, file=sys.stderr)
        raise RuntimeError(f"{description} failed with exit code {result.returncode}")


IN_COLAB = "google.colab" in sys.modules
print("Ambiente:", "Google Colab" if IN_COLAB else "Local")
print("Python   :", sys.version.split()[0])

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/almeidadm/economy-classifier.git"
    REPO_BRANCH = "main"
    DRIVE_FOLDER = "economy-classifier"

    DRIVE_BASE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    REPO_DIR = Path("/content/economy-classifier")

    if REPO_DIR.exists():
        # Pull latest so a re-run picks up commits without manual rm -rf
        _run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], "git fetch")
        _run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], "git checkout")
        _run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], "git reset")
    else:
        _run(
            ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
            "git clone",
        )

    # Install only what's missing — preserve Colab's preinstalled numpy/scipy/torch
    _run(
        [sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR),
         "--upgrade-strategy", "only-if-needed", "-q"],
        "pip install -e .",
    )

    # Make the editable install importable in the current kernel session
    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))

    SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
    if not (SPLITS_DIR / "train.parquet").exists():
        zip_path = DRIVE_BASE / "colab_splits.zip"
        assert zip_path.exists(), f"Falta colab_splits.zip em {DRIVE_BASE}. Rode scripts/colab_pack.py local e suba o zip."
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(REPO_DIR / "artifacts")
        print("Splits extraidos em", SPLITS_DIR)

    RUNS_BASE = DRIVE_BASE / "runs"
    HARDWARE = "Colab-CPU"
else:
    REPO_DIR = Path.cwd().parent
    SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
    RUNS_BASE = REPO_DIR / "artifacts" / "runs"
    HARDWARE = "Local-CPU"

RUNS_BASE.mkdir(parents=True, exist_ok=True)
print("SPLITS_DIR:", SPLITS_DIR)
print("RUNS_BASE :", RUNS_BASE)
print("HARDWARE  :", HARDWARE)


def _sanity_check_imports() -> None:
    """Fail loud at bootstrap if numpy/scipy/sklearn are inconsistent (common Colab hazard)."""
    failures = []
    for mod_name in ("numpy", "scipy", "pandas", "sklearn", "economy_classifier"):
        try:
            __import__(mod_name)
        except Exception as e:  # noqa: BLE001
            failures.append((mod_name, type(e).__name__, str(e)))
    if failures:
        print("\nImport sanity check FAILED:")
        for name, kind, msg in failures:
            print(f"  - {name}: {kind}: {msg[:200]}")
        raise RuntimeError(
            "Um ou mais modulos quebrados. Causa comum no Colab: numpy/scipy "
            "ficaram inconsistentes apos pip install. Solucao: "
            "Runtime > Restart runtime, depois execute esta celula novamente."
        )
    import numpy, scipy, pandas, sklearn  # noqa: F401, E401
    print(f"\nVersoes: numpy={numpy.__version__}  scipy={scipy.__version__}  "
          f"pandas={pandas.__version__}  sklearn={sklearn.__version__}")
    print("OK: economy_classifier importavel")


_sanity_check_imports()


## 1. Imports e configuracao

In [ ]:
import json
import time

import joblib
import numpy as np
import pandas as pd

from economy_classifier.datasets import MULTICLASS_TOP7, OTHER_LABEL, attach_multiclass_label
from economy_classifier.evaluation import (
    compute_binary_metrics,
    compute_confusion_matrix,
    compute_cost_metrics,
    compute_multiclass_metrics,
    compute_roc_auc,
    summarize_cv_metrics,
)
from economy_classifier.hyperparameter_search import (
    random_search_tfidf,
    tfidf_best_params_to_kwargs,
)
from economy_classifier.project import (
    build_result_card,
    compute_artifact_size_mb,
    persist_result_card,
)
from economy_classifier.tfidf import (
    TfidfMulticlassConfig,
    TfidfTrainingConfig,
    train_tfidf_classifier,
    train_tfidf_multiclass,
)

CLASSIFIER = "linearsvc"
MODEL_ID = "tfidf_linearsvc"
SEED = 2026
MULTI_LABELS = list(MULTICLASS_TOP7) + [OTHER_LABEL]

# Hyperparameter search budget — increase if results unstable
N_ITER_BINARY = 60
N_ITER_MULTI  = 60
CV_INNER_SPLITS = 5
CV_INNER_SEED = 2027  # different from global seed=2026 (and historical cv_folds.json seed=42) so cv_5fold reports independent variance
# n_jobs=2 evita OOM no Colab (~12 GB) com pool de 150k textos.
# Em maquinas com >=32 GB RAM voce pode subir para 4 ou -1 (todos os cores).
N_JOBS = 2


## 2. Carga dos splits e folds

In [ ]:
train = pd.read_parquet(SPLITS_DIR / "train.parquet")
val = pd.read_parquet(SPLITS_DIR / "val.parquet")
test = pd.read_parquet(SPLITS_DIR / "test.parquet")
cv_folds = json.loads((SPLITS_DIR / "cv_folds.json").read_text())

for split_name, split_df in [("train", train), ("val", val), ("test", test)]:
    if "label_multi" not in split_df.columns:
        if split_name == "train":
            train = attach_multiclass_label(train)
        elif split_name == "val":
            val = attach_multiclass_label(val)
        else:
            test = attach_multiclass_label(test)

train_val_pool = pd.concat([train, val]).sort_index()

print(f"train     : {len(train):>7,} (mercado {train['label'].mean()*100:.1f}%)")
print(f"val       : {len(val):>7,} (mercado {val['label'].mean()*100:.1f}%)")
print(f"test      : {len(test):>7,} (mercado {test['label'].mean()*100:.1f}%)")
print(f"train+val : {len(train_val_pool):>7,} (pool para search, CV e test_set)")
print(f"cv_folds  : {len(cv_folds)} folds")


## 3. Helpers de cost-tracking

In [ ]:
from economy_classifier.tfidf import get_pipeline_n_parameters as get_n_parameters

def model_size_mb(model_dir):
    return round(compute_artifact_size_mb(Path(model_dir)), 3)


def binary_metrics_from_preds(preds):
    m = compute_binary_metrics(preds["y_true"].values, preds["y_pred"].values)
    m["auc_roc"] = round(compute_roc_auc(preds["y_true"].values, preds["y_score"].values), 4)
    return m


def multiclass_metrics_from_preds(preds, labels):
    return compute_multiclass_metrics(preds["y_true"], preds["y_pred"], labels=labels)


def persist_search_log(search_result, search_dir):
    search_dir.mkdir(parents=True, exist_ok=True)
    (search_dir / "search_result.json").write_text(
        json.dumps(search_result.to_dict(), indent=2, default=str),
    )
    return search_dir / "search_result.json"


## 4. Hyperparameter search BINARIO

RandomizedSearchCV em train+val com inner StratifiedKFold(5, seed=2027). O seed do inner-CV difere do seed global (`seed=2026`) e do `cv_folds.json` (historicamente `seed=42`) para que o regime `cv_5fold` reportado adiante seja uma estimativa de variancia em folds diferentes dos usados na selecao.

In [ ]:
search_dir = RUNS_BASE / f"{MODEL_ID}_search_binary"
print(f"Hyperparameter search BINARY: {N_ITER_BINARY} trials x {CV_INNER_SPLITS}-fold inner CV")
print(f"Pool: {len(train_val_pool):,} samples (train+val)")

search_bin = random_search_tfidf(
    train_val_pool,
    classifier=CLASSIFIER,
    label_column="label",
    multiclass=False,
    n_iter=N_ITER_BINARY,
    cv_n_splits=CV_INNER_SPLITS,
    cv_seed=CV_INNER_SEED,
    n_jobs=N_JOBS,
    seed=SEED,
    verbose=2,
)

best_kwargs_bin = tfidf_best_params_to_kwargs(search_bin.best_params)
config_bin = TfidfTrainingConfig(classifier=CLASSIFIER, seed=SEED, **best_kwargs_bin)

persist_search_log(search_bin, search_dir)

print(f"\nSearch wall-clock: {search_bin.search_seconds:.1f}s")
print(f"Best inner-CV F1 : {search_bin.best_score:.4f}")
print(f"Best params      : {json.dumps(best_kwargs_bin, indent=2, default=str)}")


## 5. Hyperparameter search MULTICLASSE

Mesmo protocolo, mas otimizando `f1_macro` sobre `label_multi`.

In [ ]:
search_dir = RUNS_BASE / f"{MODEL_ID}_search_multiclass"
print(f"Hyperparameter search MULTICLASS: {N_ITER_MULTI} trials x {CV_INNER_SPLITS}-fold inner CV")

search_multi = random_search_tfidf(
    train_val_pool,
    classifier=CLASSIFIER,
    label_column="label_multi",
    multiclass=True,
    strategy="native",
    n_iter=N_ITER_MULTI,
    cv_n_splits=CV_INNER_SPLITS,
    cv_seed=CV_INNER_SEED,
    n_jobs=N_JOBS,
    seed=SEED,
    verbose=2,
)

best_kwargs_multi = tfidf_best_params_to_kwargs(search_multi.best_params)
config_multi = TfidfMulticlassConfig(
    classifier=CLASSIFIER, strategy="native", seed=SEED, **best_kwargs_multi,
)

persist_search_log(search_multi, search_dir)

print(f"\nSearch wall-clock: {search_multi.search_seconds:.1f}s")
print(f"Best inner-CV macro_F1: {search_multi.best_score:.4f}")
print(f"Best params           : {json.dumps(best_kwargs_multi, indent=2, default=str)}")


## 6. BINARIO — `fixed_split` (treina train com best_params, avalia val)

In [ ]:
run_dir = RUNS_BASE / f"{MODEL_ID}_binary_fixed_split"
run_dir.mkdir(parents=True, exist_ok=True)

result = train_tfidf_classifier(train, val, run_dir=run_dir, config=config_bin)
metrics = binary_metrics_from_preds(result["predictions"])
pipeline = joblib.load(Path(result["model_dir"]) / "tfidf_pipeline.joblib")
cost = compute_cost_metrics(
    train_seconds=result["timing"]["train_seconds"],
    inference_seconds=result["timing"]["inference_seconds"],
    n_inference_samples=len(val),
    model_size_mb=model_size_mb(result["model_dir"]),
    n_parameters=get_n_parameters(pipeline),
    hardware=HARDWARE,
)
result["predictions"].to_csv(run_dir / "predictions.csv", index=False)
card = build_result_card(
    model_id=MODEL_ID, task="binary", regime="fixed_split",
    metrics=metrics, cost=cost, config=config_bin.to_dict(),
    n_train_samples=len(train), n_eval_samples=len(val),
    predictions_path=str(run_dir / "predictions.csv"),
    hyperparameter_search=search_bin.card_payload(),
)
persist_result_card(card, run_dir)
print("BINARY fixed_split:", json.dumps({"metrics": metrics, "cost": cost}, indent=2))


## 7. BINARIO — `cv_5fold` (best_params em 5 folds independentes)

In [ ]:
run_dir = RUNS_BASE / f"{MODEL_ID}_binary_cv_5fold"
run_dir.mkdir(parents=True, exist_ok=True)

fold_metrics, train_times, inf_times = [], [], []
all_preds = []
for i, fold in enumerate(cv_folds):
    fold_dir = run_dir / f"fold_{i}"
    tr = train_val_pool.loc[fold["train_indices"]]
    va = train_val_pool.loc[fold["val_indices"]]
    res = train_tfidf_classifier(tr, va, run_dir=fold_dir, config=config_bin)
    fm = binary_metrics_from_preds(res["predictions"])
    fold_metrics.append(fm)
    train_times.append(res["timing"]["train_seconds"])
    inf_times.append(res["timing"]["inference_seconds"])
    res["predictions"]["fold"] = i
    all_preds.append(res["predictions"])
    print(f"  fold {i}: f1={fm['f1']:.4f}  auc={fm['auc_roc']:.4f}")

summary = summarize_cv_metrics(fold_metrics)
last_pipeline = joblib.load(Path(res["model_dir"]) / "tfidf_pipeline.joblib")
cost = compute_cost_metrics(
    train_seconds=train_times, inference_seconds=inf_times,
    n_inference_samples=len(train_val_pool) // len(cv_folds),
    model_size_mb=model_size_mb(res["model_dir"]),
    n_parameters=get_n_parameters(last_pipeline),
    hardware=HARDWARE,
)
pd.concat(all_preds).to_csv(run_dir / "predictions.csv", index=False)
card = build_result_card(
    model_id=MODEL_ID, task="binary", regime="cv_5fold",
    metrics=summary, cost=cost, config=config_bin.to_dict(),
    n_train_samples=len(train_val_pool), n_eval_samples=len(train_val_pool),
    predictions_path=str(run_dir / "predictions.csv"),
    notes="5-fold stratified CV on the 90% train+val pool (folds independent of search inner CV).",
    hyperparameter_search=search_bin.card_payload(),
)
persist_result_card(card, run_dir)
print(f"\nBINARY cv_5fold: f1 = {summary['f1_mean']:.4f} +/- {summary['f1_std']:.4f}")


## 8. BINARIO — `test_set` (best_params, treina train+val, avalia teste FIXO)

In [ ]:
run_dir = RUNS_BASE / f"{MODEL_ID}_binary_test_set"
run_dir.mkdir(parents=True, exist_ok=True)

result = train_tfidf_classifier(train_val_pool, test, run_dir=run_dir, config=config_bin)
metrics = binary_metrics_from_preds(result["predictions"])
pipeline = joblib.load(Path(result["model_dir"]) / "tfidf_pipeline.joblib")
cost = compute_cost_metrics(
    train_seconds=result["timing"]["train_seconds"],
    inference_seconds=result["timing"]["inference_seconds"],
    n_inference_samples=len(test),
    model_size_mb=model_size_mb(result["model_dir"]),
    n_parameters=get_n_parameters(pipeline),
    hardware=HARDWARE,
)
result["predictions"].to_csv(run_dir / "predictions.csv", index=False)
card = build_result_card(
    model_id=MODEL_ID, task="binary", regime="test_set",
    metrics=metrics, cost=cost, config=config_bin.to_dict(),
    n_train_samples=len(train_val_pool), n_eval_samples=len(test),
    predictions_path=str(run_dir / "predictions.csv"),
    notes="Final single-shot evaluation on the held-out 10% test set, refit on train+val.",
    hyperparameter_search=search_bin.card_payload(),
)
persist_result_card(card, run_dir)
print("BINARY test_set:", json.dumps({"metrics": metrics, "cost": cost}, indent=2))


## 9. MULTICLASSE (native softmax) — `fixed_split`

In [ ]:
run_dir = RUNS_BASE / f"{MODEL_ID}_multiclass_fixed_split"
run_dir.mkdir(parents=True, exist_ok=True)

result = train_tfidf_multiclass(train, val, label_column="label_multi", run_dir=run_dir, config=config_multi)
metrics = multiclass_metrics_from_preds(result["predictions"], labels=MULTI_LABELS)
pipeline = joblib.load(Path(result["model_dir"]) / "tfidf_pipeline.joblib")
cost = compute_cost_metrics(
    train_seconds=result["timing"]["train_seconds"],
    inference_seconds=result["timing"]["inference_seconds"],
    n_inference_samples=len(val),
    model_size_mb=model_size_mb(result["model_dir"]),
    n_parameters=get_n_parameters(pipeline),
    hardware=HARDWARE,
)
result["predictions"].to_csv(run_dir / "predictions.csv", index=False)
cm = compute_confusion_matrix(result["predictions"]["y_true"], result["predictions"]["y_pred"], labels=MULTI_LABELS)
cm.to_csv(run_dir / "confusion_matrix.csv")
card = build_result_card(
    model_id=MODEL_ID, task="multiclass", regime="fixed_split",
    metrics=metrics, cost=cost, config=config_multi.to_dict(),
    n_train_samples=len(train), n_eval_samples=len(val),
    predictions_path=str(run_dir / "predictions.csv"),
    hyperparameter_search=search_multi.card_payload(),
)
persist_result_card(card, run_dir)
print("MULTI fixed_split:", json.dumps({k: v for k, v in metrics.items() if k != 'per_class_f1'}, indent=2))


## 10. MULTICLASSE — `cv_5fold`

In [ ]:
run_dir = RUNS_BASE / f"{MODEL_ID}_multiclass_cv_5fold"
run_dir.mkdir(parents=True, exist_ok=True)

fold_metrics, train_times, inf_times = [], [], []
all_preds = []
for i, fold in enumerate(cv_folds):
    fold_dir = run_dir / f"fold_{i}"
    tr = train_val_pool.loc[fold["train_indices"]]
    va = train_val_pool.loc[fold["val_indices"]]
    res = train_tfidf_multiclass(tr, va, label_column="label_multi", run_dir=fold_dir, config=config_multi)
    fm = multiclass_metrics_from_preds(res["predictions"], labels=MULTI_LABELS)
    fold_metrics.append(fm)
    train_times.append(res["timing"]["train_seconds"])
    inf_times.append(res["timing"]["inference_seconds"])
    res["predictions"]["fold"] = i
    all_preds.append(res["predictions"])
    print(f"  fold {i}: macro_f1={fm['macro_f1']:.4f}  acc={fm['accuracy']:.4f}")

summary = summarize_cv_metrics(fold_metrics)
last_pipeline = joblib.load(Path(res["model_dir"]) / "tfidf_pipeline.joblib")
cost = compute_cost_metrics(
    train_seconds=train_times, inference_seconds=inf_times,
    n_inference_samples=len(train_val_pool) // len(cv_folds),
    model_size_mb=model_size_mb(res["model_dir"]),
    n_parameters=get_n_parameters(last_pipeline),
    hardware=HARDWARE,
)
pd.concat(all_preds).to_csv(run_dir / "predictions.csv", index=False)
card = build_result_card(
    model_id=MODEL_ID, task="multiclass", regime="cv_5fold",
    metrics=summary, cost=cost, config=config_multi.to_dict(),
    n_train_samples=len(train_val_pool), n_eval_samples=len(train_val_pool),
    predictions_path=str(run_dir / "predictions.csv"),
    notes="5-fold stratified CV on the 90% train+val pool (folds independent of search inner CV).",
    hyperparameter_search=search_multi.card_payload(),
)
persist_result_card(card, run_dir)
print(f"\nMULTI cv_5fold: macro_f1 = {summary['macro_f1_mean']:.4f} +/- {summary['macro_f1_std']:.4f}")


## 11. MULTICLASSE — `test_set` (single shot final)

In [ ]:
run_dir = RUNS_BASE / f"{MODEL_ID}_multiclass_test_set"
run_dir.mkdir(parents=True, exist_ok=True)

result = train_tfidf_multiclass(train_val_pool, test, label_column="label_multi", run_dir=run_dir, config=config_multi)
metrics = multiclass_metrics_from_preds(result["predictions"], labels=MULTI_LABELS)
pipeline = joblib.load(Path(result["model_dir"]) / "tfidf_pipeline.joblib")
cost = compute_cost_metrics(
    train_seconds=result["timing"]["train_seconds"],
    inference_seconds=result["timing"]["inference_seconds"],
    n_inference_samples=len(test),
    model_size_mb=model_size_mb(result["model_dir"]),
    n_parameters=get_n_parameters(pipeline),
    hardware=HARDWARE,
)
result["predictions"].to_csv(run_dir / "predictions.csv", index=False)
cm = compute_confusion_matrix(result["predictions"]["y_true"], result["predictions"]["y_pred"], labels=MULTI_LABELS)
cm.to_csv(run_dir / "confusion_matrix.csv")
card = build_result_card(
    model_id=MODEL_ID, task="multiclass", regime="test_set",
    metrics=metrics, cost=cost, config=config_multi.to_dict(),
    n_train_samples=len(train_val_pool), n_eval_samples=len(test),
    predictions_path=str(run_dir / "predictions.csv"),
    notes="Final single-shot evaluation on the held-out 10% test set, refit on train+val.",
    hyperparameter_search=search_multi.card_payload(),
)
persist_result_card(card, run_dir)
print("MULTI test_set:", json.dumps({k: v for k, v in metrics.items() if k != 'per_class_f1'}, indent=2))


## 12. Sumario — 6 result cards + 2 logs de busca

In [ ]:
rows = []
for sub in sorted(RUNS_BASE.glob(f"{MODEL_ID}_*")):
    card_path = sub / "result_card.json"
    if not card_path.exists():
        continue
    c = json.loads(card_path.read_text())
    if c.get("task") not in ("binary", "multiclass"):
        continue
    metrics = c["metrics"]
    primary = (
        metrics.get("f1") or metrics.get("f1_mean")
        or metrics.get("macro_f1") or metrics.get("macro_f1_mean")
    )
    rows.append({
        "task": c["task"], "regime": c["regime"],
        "primary": primary,
        "train_s": c["cost"].get("train_seconds_mean"),
        "inf_s":   c["cost"].get("inference_seconds_mean"),
        "size_mb": c["cost"].get("model_size_mb"),
        "params":  c["cost"].get("n_parameters"),
        "search_s": (c.get("hyperparameter_search") or {}).get("search_seconds"),
        "n_trials": (c.get("hyperparameter_search") or {}).get("n_trials"),
    })
summary_df = pd.DataFrame(rows).sort_values(["task", "regime"]).reset_index(drop=True)
summary_df
